In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw_books_with_genre.csv")

In [3]:
df.head()

,title,price,availability,rating_word,genre
0,It's Only the Himalayas,Â£45.17,In stock,Two,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,In stock,Four,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,In stock,Three,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,In stock,Two,Travel
4,Under the Tuscan Sun,Â£37.33,In stock,Three,Travel


In [4]:
df.shape

(781, 5)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         781 non-null    object
 1   price         781 non-null    object
 2   availability  781 non-null    object
 3   rating_word   781 non-null    object
 4   genre         781 non-null    object
dtypes: object(5)
memory usage: 30.6+ KB


In [6]:
df.isna().sum()

title           0
price           0
availability    0
rating_word     0
genre           0
dtype: int64

In [7]:
df["price"] = (
    df["price"]
    .str.replace("Â£", "", regex=False)
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [8]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["rating_word"].map(rating_map)

In [9]:
df.drop(columns="rating_word", inplace=True)

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         781 non-null    object 
 1   price         781 non-null    float64
 2   availability  781 non-null    object 
 3   genre         781 non-null    object 
 4   rating        781 non-null    int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 30.6+ KB


In [11]:
df["title"] = (
    df["title"]
    .str.replace('Â', '', regex=False)
    .str.replace('�', '', regex=False)
    .str.replace('"', '', regex=False)
    .str.strip()
)

In [12]:
import re
df["title"] = df["title"].apply(
    lambda x: re.sub(r"[^\x00-\x7F]+", "", x)
)

In [13]:
df["title"]

0                                It's Only the Himalayas
1      Full Moon over Noahs Ark: An Odyssey to Mount ...
2      See America: A Celebration of Our National Par...
3      Vagabonding: An Uncommon Guide to the Art of L...
4                                   Under the Tuscan Sun
                             ...                        
776    Why the Right Went Wrong: Conservatism--From G...
777    Equal Is Unfair: America's Misguided Fight Aga...
778                                       Amid the Chaos
779                                           Dark Notes
780    The Long Shadow of Small Ghosts: Murder and Me...
Name: title, Length: 781, dtype: object

In [14]:
df["genre"].value_counts()

genre
Nonfiction            110
Sequential Art         75
Fiction                65
Young Adult            54
Fantasy                48
Romance                35
Mystery                32
Food and Drink         30
Childrens              29
Historical Fiction     26
Classics               19
Poetry                 19
History                18
Womens Fiction         17
Horror                 17
Science Fiction        16
Science                14
Music                  13
Business               12
Thriller               11
Philosophy             11
Travel                 11
Humor                  10
Autobiography           9
Art                     8
Psychology              7
Religion                7
Christian Fiction       6
New Adult               6
Spirituality            6
Self Help               5
Sports and Games        5
Biography               5
Health                  4
Contemporary            3
Politics                3
Christian               3
Historical              2
Cultur

In [15]:
def map_genre_custom(genre):
    # 1. Non-Fiction (General)
    if genre in [
        "Nonfiction", "Business", "Biography"
    ]:
        return "Non-Fiction (General)"

    # 2. History / Psychology / Human Stories (includes Sequential Art)
    elif genre in [
        "History", "Psychology", "Autobiography", "Self Help", "Sequential Art"
    ]:
        return "History / Psychology / Human Stories"

    # 3. Core Fiction
    elif genre in [
        "Fiction", "Mystery", "Historical Fiction",
        "Womens Fiction", "Science Fiction", "Christian Fiction"
    ]:
        return "Core Fiction"

    # 4. Speculative / Intense Fiction
    elif genre in [
        "Fantasy", "Horror", "Science", "Thriller"
    ]:
        return "Speculative / Intense Fiction"

    # 5. Lifestyle & Light Reads
    elif genre in [
        "Food and Drink", "Travel", "Humor", "Sports and Games", "Health"
    ]:
        return "Lifestyle & Light Reads"

    # 6. Young Adult / New Adult
    elif genre in [
        "Young Adult", "New Adult"
    ]:
        return "Young Adult / New Adult"

    # 7. Romance
    elif genre == "Romance":
        return "Romance"

    # 8. Children
    elif genre == "Childrens":
        return "Children"

    # 9. Arts / Philosophy / Spiritual
    elif genre in [
        "Classics", "Poetry", "Music",
        "Philosophy", "Art", "Religion", "Spirituality"
    ]:
        return "Arts / Philosophy / Spiritual"

    # 10. Other
    else:
        return "Other"

In [16]:
df["genre_group"] = df["genre"].apply(map_genre_custom)

In [17]:
df["genre_group"].value_counts()

genre_group
Core Fiction                            162
Non-Fiction (General)                   127
History / Psychology / Human Stories    114
Speculative / Intense Fiction            90
Arts / Philosophy / Spiritual            83
Lifestyle & Light Reads                  60
Young Adult / New Adult                  60
Romance                                  35
Children                                 29
Other                                    21
Name: count, dtype: int64

In [18]:
df["in_stock"] = df["availability"].str.lower().eq("in stock")

In [19]:
df.drop(columns="availability", inplace=True)

In [20]:
df["title"].isna().sum()

0

In [21]:
df["title"].duplicated().sum()

1

In [22]:
df = df.drop_duplicates(subset="title")

In [23]:
df.describe()["price"]

count    780.000000
mean      35.143590
std       14.516895
min       10.000000
25%       22.110000
50%       35.940000
75%       47.820000
max       59.990000
Name: price, dtype: float64

In [24]:
df["genre"] = df["genre"].str.strip().str.title()

In [25]:
# Absolute price bands (storytelling)
df["price_band"] = pd.cut(
    df["price"],
    bins=[0, 15, 30, 45, 60],
    labels=["Low", "Lower-Mid", "Mid", "Upper-Mid"]
)

# Relative price tiers (analysis)
df["price_tier"] = pd.qcut(
    df["price"],
    q=4,
    labels=["Cheapest", "Mid-Low", "Mid-High", "Most Expensive"]
)

In [26]:
df["rating_label"] = df["rating"].map({
    1: "Poor", 2: "Average", 3: "Good", 4: "Very Good", 5: "Excellent"
})

In [27]:
df.head(10)

,title,price,genre,rating,genre_group,in_stock,price_band,price_tier,rating_label
0,It's Only the Himalayas,45.17,Travel,2,Lifestyle & Light Reads,True,Upper-Mid,Mid-High,Average
1,Full Moon over Noahs Ark: An Odyssey to Mount ...,49.43,Travel,4,Lifestyle & Light Reads,True,Upper-Mid,Most Expensive,Very Good
2,See America: A Celebration of Our National Par...,48.87,Travel,3,Lifestyle & Light Reads,True,Upper-Mid,Most Expensive,Good
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Travel,2,Lifestyle & Light Reads,True,Mid,Mid-High,Average
4,Under the Tuscan Sun,37.33,Travel,3,Lifestyle & Light Reads,True,Mid,Mid-High,Good
5,A Summer In Europe,44.34,Travel,2,Lifestyle & Light Reads,True,Mid,Mid-High,Average
6,The Great Railway Bazaar,30.54,Travel,1,Lifestyle & Light Reads,True,Mid,Mid-Low,Poor
7,A Year in Provence (Provence #1),56.88,Travel,4,Lifestyle & Light Reads,True,Upper-Mid,Most Expensive,Very Good
8,The Road to Little Dribbling: Adventures of an...,23.21,Travel,1,Lifestyle & Light Reads,True,Lower-Mid,Mid-Low,Poor
9,Neither Here nor There: Travels in Europe,38.95,Travel,3,Lifestyle & Light Reads,True,Mid,Mid-High,Good


In [28]:
df.sample(10)

,title,price,genre,rating,genre_group,in_stock,price_band,price_tier,rating_label
703,"Naturally Lean: 125 Nourishing Gluten-Free, Pl...",11.38,Food And Drink,5,Lifestyle & Light Reads,True,Low,Cheapest,Excellent
585,The Origin of Species,10.01,Science,4,Speculative / Intense Fiction,True,Low,Cheapest,Very Good
266,Deception Point,40.32,Fiction,4,Core Fiction,True,Mid,Mid-High,Very Good
470,Icing (Aces Hockey #2),40.44,Sports And Games,4,Lifestyle & Light Reads,True,Mid,Mid-High,Very Good
270,Siddhartha,34.22,Fiction,5,Core Fiction,True,Mid,Mid-Low,Excellent
701,The Moosewood Cookbook: Recipes from Moosewood...,12.34,Food And Drink,4,Lifestyle & Light Reads,True,Low,Cheapest,Very Good
91,"Fables, Vol. 1: Legends in Exile (Fables #1)",41.62,Sequential Art,4,History / Psychology / Human Stories,True,Mid,Mid-High,Very Good
105,Art Ops Vol. 1,48.80,Sequential Art,3,History / Psychology / Human Stories,True,Upper-Mid,Most Expensive,Good
482,The Hidden Oracle (The Trials of Apollo #1),52.26,Fantasy,2,Speculative / Intense Fiction,True,Upper-Mid,Most Expensive,Average
291,Birdsong: A Story in Pictures,54.64,Childrens,3,Children,True,Upper-Mid,Most Expensive,Good


In [29]:
df.to_csv("../data/books_clean_final.csv", index=False)

In [30]:
df.head()

,title,price,genre,rating,genre_group,in_stock,price_band,price_tier,rating_label
0,It's Only the Himalayas,45.17,Travel,2,Lifestyle & Light Reads,True,Upper-Mid,Mid-High,Average
1,Full Moon over Noahs Ark: An Odyssey to Mount ...,49.43,Travel,4,Lifestyle & Light Reads,True,Upper-Mid,Most Expensive,Very Good
2,See America: A Celebration of Our National Par...,48.87,Travel,3,Lifestyle & Light Reads,True,Upper-Mid,Most Expensive,Good
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,Travel,2,Lifestyle & Light Reads,True,Mid,Mid-High,Average
4,Under the Tuscan Sun,37.33,Travel,3,Lifestyle & Light Reads,True,Mid,Mid-High,Good
